# 05 - Model Evaluation
## Оцінка моделей та метрики (2e)

У цьому notebook ми:
- Завантажимо навчені моделі
- Оцінимо їх продуктивність
- Побудуємо confusion matrices
- Згенеруємо classification reports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.embeddings import load_embeddings
from src.train import load_model, split_data
from src.evaluate import (
    evaluate_model,
    print_evaluation_results,
    plot_confusion_matrix,
    generate_classification_report
)
from src.config import PROCESSED_DATA_PATH, MODELS_PATH

%matplotlib inline

## 1. Завантаження даних

In [ ]:
# Завантажуємо дані та ембединги
df = pd.read_csv(PROCESSED_DATA_PATH / 'cleaned_data.csv')
embeddings = load_embeddings(str(PROCESSED_DATA_PATH / 'embeddings.npy'))

# Розділяємо на train/test (використовуємо той самий split)
X = embeddings
y = df['label'].values
X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Test size: {len(X_test)}")

## 2. Оцінка Logistic Regression

In [ ]:
# Завантажимо модель
lr_model = load_model(str(MODELS_PATH / 'logistic_regression.joblib'))

# Зробимо передбачення
y_pred_lr = lr_model.predict(X_test)

# Оцінимо модель
print("=" * 60)
print("Logistic Regression")
print("=" * 60)

metrics_lr = evaluate_model(y_test, y_pred_lr)
print_evaluation_results(metrics_lr)

In [ ]:
# Confusion matrix
plot_confusion_matrix(
    y_test,
    y_pred_lr,
    labels=['Real', 'Fake'],
    save=True,
    filename='confusion_matrix_lr.png'
)

In [ ]:
# Classification report
report_lr = generate_classification_report(
    y_test,
    y_pred_lr,
    target_names=['Real', 'Fake'],
    save=True,
    filename='classification_report_lr.txt'
)

## 3. Оцінка Random Forest

In [ ]:
# Завантажимо модель
rf_model = load_model(str(MODELS_PATH / 'random_forest.joblib'))

# Зробимо передбачення
y_pred_rf = rf_model.predict(X_test)

# Оцінимо модель
print("=" * 60)
print("Random Forest")
print("=" * 60)

metrics_rf = evaluate_model(y_test, y_pred_rf)
print_evaluation_results(metrics_rf)

In [ ]:
# Confusion matrix
plot_confusion_matrix(
    y_test,
    y_pred_rf,
    labels=['Real', 'Fake'],
    save=True,
    filename='confusion_matrix_rf.png'
)

In [ ]:
# Classification report
report_rf = generate_classification_report(
    y_test,
    y_pred_rf,
    target_names=['Real', 'Fake'],
    save=True,
    filename='classification_report_rf.txt'
)

## 4. Оцінка SVM

In [ ]:
# Завантажимо модель
svm_model = load_model(str(MODELS_PATH / 'svm.joblib'))

# Зробимо передбачення
y_pred_svm = svm_model.predict(X_test)

# Оцінимо модель
print("=" * 60)
print("SVM")
print("=" * 60)

metrics_svm = evaluate_model(y_test, y_pred_svm)
print_evaluation_results(metrics_svm)

In [ ]:
# Confusion matrix
plot_confusion_matrix(
    y_test,
    y_pred_svm,
    labels=['Real', 'Fake'],
    save=True,
    filename='confusion_matrix_svm.png'
)

In [ ]:
# Classification report
report_svm = generate_classification_report(
    y_test,
    y_pred_svm,
    target_names=['Real', 'Fake'],
    save=True,
    filename='classification_report_svm.txt'
)

## 5. Порівняння моделей

In [ ]:
# Створимо DataFrame з результатами
comparison_df = pd.DataFrame({
    'Logistic Regression': metrics_lr,
    'Random Forest': metrics_rf,
    'SVM': metrics_svm
}).T

print("\nПорівняння моделей:")
print(comparison_df)

In [ ]:
# Візуалізуємо порівняння
fig, ax = plt.subplots(figsize=(12, 6))

comparison_df.plot(kind='bar', ax=ax)
plt.title('Model Performance Comparison on Test Set', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)
plt.xlabel('Model', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

from src.config import OUTPUTS_PATH
plt.savefig(OUTPUTS_PATH / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Аналіз помилок

In [ ]:
# Знайдемо помилково класифіковані зразки (на прикладі Logistic Regression)
misclassified_indices = np.where(y_test != y_pred_lr)[0]

print(f"Кількість помилок: {len(misclassified_indices)}")
print(f"Відсоток помилок: {len(misclassified_indices) / len(y_test) * 100:.2f}%")

# Подивимось на кілька прикладів помилок
print("\nПриклади помилкової класифікації:")
test_df = df.iloc[len(X_train):].reset_index(drop=True)

for i in misclassified_indices[:3]:
    print(f"\n--- Помилка {i+1} ---")
    print(f"Правильна мітка: {y_test[i]}")
    print(f"Передбачена мітка: {y_pred_lr[i]}")
    print(f"Текст: {test_df.iloc[i]['text'][:200]}...")

## Висновки

✅ Оцінено всі навчені моделі на тестових даних
✅ Побудовано confusion matrices
✅ Згенеровано classification reports
✅ Порівняно продуктивність моделей
✅ Проаналізовано помилки класифікації